# Módulo 2 · Ingesta a RAW

⏱ **8 min** · Slides: 3, 6 (una sola copia, almacenamiento abierto) y 27 (primer paso de la ingesta)

**Objetivo:** traer datos reales a Databricks y dejarlos **sin tocar** en un Volumen.

**Fuente:** API de TVMaze · `GET https://api.tvmaze.com/schedule/web?date=AAAA-MM-DD`
→ un JSON con ~400 episodios por día.

```
API TVMaze ──► /Volumes/.../raw/tvmaze/AAAA/MM/DD/tvmaze.json   (un archivo por día)
```

| Paso | Qué haces |
|---|---|
| 1 | Probar la API con un día |
| 2 | Definir parámetros (widgets) |
| 3 | Descargar varios días a RAW |
| 4 | Verificar lo descargado |

> **Si tu workspace no tiene salida a internet:** reutiliza los archivos de la práctica `01_ELT`
> (`/Volumes/workspace/raw/tvmze`) cambiando `RAW_PATH` en `00_Setup`, y salta al módulo 3.

In [0]:
%run ./00_Setup

## Paso 1 · Probar la API con un día

In [0]:
import requests

r = requests.get("https://api.tvmaze.com/schedule/web", params={"date": "2024-03-01"}, timeout=30)
episodios = r.json()

print("status   :", r.status_code)
print("episodios:", len(episodios))
print("campos   :", list(episodios[0].keys()))
print(episodios)


👀 **Qué observar:** `_embedded` trae la serie dentro de cada episodio. Es un JSON **anidado**.
Lo trabajaremos con PySpark en el módulo 3.

## Paso 2 · Parámetros con widgets

In [0]:
dbutils.widgets.text("start_date", "2024-03-01", "Fecha inicial (AAAA-MM-DD)")
dbutils.widgets.text("dias", "7", "Días a descargar")

## Paso 3 · Descargar a RAW

Dos reglas de RAW:
1. **No se transforma**: se guarda el JSON tal cual llega.
2. **Idempotente**: si el archivo ya existe, no se vuelve a bajar.

In [0]:
from datetime import date, timedelta
from pathlib import Path

inicio = date.fromisoformat(dbutils.widgets.get("start_date"))
dias   = int(dbutils.widgets.get("dias"))

descargados, omitidos, fallidos = [], [], []

for i in range(dias):
    fecha   = inicio + timedelta(days=i)
    destino = Path(RAW_PATH) / f"{fecha:%Y}" / f"{fecha:%m}" / f"{fecha:%d}" / "tvmaze.json"

    if destino.exists():
        omitidos.append(fecha)
        continue

    try:
        resp = requests.get("https://api.tvmaze.com/schedule/web",
                            params={"date": fecha.isoformat()}, timeout=30)
        resp.raise_for_status()
    except requests.RequestException:
        fallidos.append(fecha)
        continue

    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(resp.content)          # el JSON exactamente como lo devolvió la API
    descargados.append(fecha)
    time.sleep(0.5)                            # cortesía con la API

print(f"Descargados: {len(descargados)} | Ya existían: {len(omitidos)} | Fallidos: {len(fallidos)}")

🧪 **Pruébalo:** vuelve a ejecutar la celda anterior. Ahora todos aparecen como "Ya existían".
Eso es una carga **idempotente**.

## Paso 4 · Verificar lo descargado

In [0]:
archivos = spark.read.format("binaryFile").load(RAW_GLOB)
display(archivos.select("path", "length", "modificationTime").orderBy("path"))

In [0]:
print("Archivos:", archivos.count())
print("Tamaño total:", size_mb(RAW_PATH), "MB")

In [0]:
# Primeros caracteres de uno de los archivos
primero = archivos.orderBy("path").first()["path"].replace("dbfs:", "")
print(dbutils.fs.head(primero, 600))

👀 **Qué observar:** es texto JSON crudo. Aún no hay columnas ni tipos: eso lo aporta el DataFrame.
Revisa también **Catalog → lab_tvmaze → datos**: la carpeta `raw/tvmaze` ya tiene la jerarquía año/mes/día.

## ✔ Checkpoint

- [ ] Hay un `tvmaze.json` por día en el Volumen
- [ ] Entiendo por qué RAW no se transforma (si algo falla, se reconstruye desde aquí)
- [ ] Re-ejecutar la descarga no duplica nada

**Siguiente:** `03_DataFrames`